# Deep Mining and Biostatistical Analysis of Marine Enzymes from Metagenomes

This Jupyter Notebook integrates the entire pipeline for statistical data analysis, biochemical property calculation, and ecological visualization following the enzyme vs. non-enzyme classification of protein sequences from 16,240 marine Metagenome-Assembled Genomes (MAGs).

**Core Data Scale**:
- Number of Metagenome-Assembled Genomes (MAGs): 16,240
- Total Predicted Protein Sequences: 33,479,647
- Total Predicted Enzyme Sequences: 17,492,382

## 1. Import Necessary Libraries and Global Configurations
Configuring plotting parameters to adhere to Science/Nature journal standards.

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from scipy import stats
from tqdm.notebook import tqdm
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# =============================================================================
# Science Journal Style Configurations
# =============================================================================
mpl.rcParams['pdf.fonttype'] = 42       # Ensures fonts are embedded in PDF
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['axes.linewidth'] = 1.2
mpl.rcParams['xtick.major.width'] = 1.2
mpl.rcParams['ytick.major.width'] = 1.2
mpl.rcParams['xtick.labelsize'] = 11
mpl.rcParams['ytick.labelsize'] = 11
mpl.rcParams['axes.labelsize'] = 13
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['legend.fontsize'] = 11
mpl.rcParams['legend.frameon'] = False  # No box around legend

sns.set_theme(style="ticks", context="paper", font_scale=1.3)

# Path Configuration (Modify according to your actual environment)
PRED_DIR = "/work1/dhy_data/MAG_43191/enzyme_predictions"
OUT_DIR = "/work1/dhy_data/MAG_43191/biostat_report"
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Parse Prediction Results and Extract Macroscopic Ecological Data
Read all CSV prediction results, and calculate the total number of sequences, enzyme ratio, and prediction probability distribution for each MAG.

In [ ]:
csv_files = glob.glob(os.path.join(PRED_DIR, "*.csv"))
print(f"Found {len(csv_files)} MAG prediction result files.")

mag_ratios = []
mag_total_seqs = []
mag_mean_probs = []
all_probabilities = [] # Collect a fraction of probabilities for the global distribution plot

for f in tqdm(csv_files, desc="Parsing CSVs"):
    try:
        df = pd.read_csv(f, usecols=["PredLabel", "Prob_Enzyme"])
        total_seqs = len(df)
        if total_seqs == 0:
            continue
            
        is_enzyme = (df["PredLabel"] == "Enzyme")
        enzyme_count = is_enzyme.sum()
        ratio = enzyme_count / total_seqs
        
        if enzyme_count > 0:
            mean_prob = df.loc[is_enzyme, "Prob_Enzyme"].mean()
            # To avoid memory overflow, sample the probabilities proportionally
            sampled_probs = df.loc[is_enzyme, "Prob_Enzyme"].sample(frac=0.01, random_state=42).tolist()
            all_probabilities.extend(sampled_probs)
        else:
            mean_prob = 0.0
            
        mag_ratios.append(ratio * 100) # Convert to percentage
        mag_total_seqs.append(total_seqs)
        mag_mean_probs.append(mean_prob)
    except Exception:
        pass

mag_ratios = np.array(mag_ratios)
mag_total_seqs = np.array(mag_total_seqs)
mag_enzyme_counts = mag_total_seqs * (mag_ratios / 100.0)
mag_mean_probs = np.array(mag_mean_probs)
all_probabilities = np.array(all_probabilities)

print(f"\nStatistics complete. Mean enzyme ratio: {np.mean(mag_ratios):.2f}% | Median: {np.median(mag_ratios):.2f}%")

## 3. Generate Top-Tier Journal Style Panel Plot (Panels A, B, C)
Plotting the Metabolic Scaling Law, Bivariate Density Contour Map, and Ecological Strategy Boxplot.

In [ ]:
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3, wspace=0.25)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[0, 2])

for ax, label in zip([axA, axB, axC], ['A', 'B', 'C']):
    ax.text(-0.15, 1.05, label, transform=ax.transAxes, 
            fontsize=20, fontweight='bold', va='bottom', ha='right')

# -------------------------------------------------------------------------
# Panel A: Metabolic Scaling Law (Scatter + Regression)
# -------------------------------------------------------------------------
r, p_value = stats.pearsonr(mag_total_seqs, mag_enzyme_counts)
p_str = "P < 1e-10" if p_value < 1e-10 else f"P = {p_value:.2e}"

scatter_color = '#3C5488' # Dark Blue
line_color = '#E64B35'    # Red

axA.scatter(mag_total_seqs, mag_enzyme_counts, alpha=0.15, color=scatter_color, s=12, edgecolors='none')
z = np.polyfit(mag_total_seqs, mag_enzyme_counts, 1)
p = np.poly1d(z)
x_trend = np.linspace(mag_total_seqs.min(), mag_total_seqs.max(), 100)
axA.plot(x_trend, p(x_trend), color=line_color, linewidth=2.5, label=f"R² = {r**2:.3f}\n{p_str}")
axA.set_xlabel('Total ORFs (Genome Size Proxy)')
axA.set_ylabel('Number of Enzyme ORFs')
axA.legend(loc='upper left', handlelength=1)

# -------------------------------------------------------------------------
# Panel B: Bivariate Density (KDE Contour)
# -------------------------------------------------------------------------
mask = mag_total_seqs < np.percentile(mag_total_seqs, 99.5)
sns.kdeplot(
    x=mag_total_seqs[mask], y=mag_ratios[mask],
    cmap="viridis", fill=True, thresh=0.05, levels=12, ax=axB
)
axB.set_xlabel('Total ORFs (Genome Size Proxy)')
axB.set_ylabel('Enzyme Proportion (%)')

# -------------------------------------------------------------------------
# Panel C: Ecological Strategy Boxplot
# -------------------------------------------------------------------------
categories = []
for r_val in mag_ratios:
    if r_val < 40:
        categories.append("Streamlined\n(<40%)")
    elif r_val > 60:
        categories.append("Generalist\n(>60%)")
    else:
        categories.append("Intermediate\n(40-60%)")
        
df_box = pd.DataFrame({"Genome_Size": mag_total_seqs, "Strategy": categories})
order = ["Streamlined\n(<40%)", "Intermediate\n(40-60%)", "Generalist\n(>60%)"]

sns.boxplot(
    x="Strategy", y="Genome_Size", hue="Strategy", data=df_box, 
    order=order, palette=["#4DBBD5", "#00A087", "#E64B35"],
    showfliers=False, width=0.5, linewidth=1.5, ax=axC, legend=False
)
axC.set_xlabel('Ecological Strategy')
axC.set_ylabel('Total Number of ORFs')

for ax in [axA, axB, axC]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=5, width=1.2)

plt.savefig(os.path.join(OUT_DIR, "Science_Panel_Ecological_Insights.pdf"), dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## 4. Biochemical Properties Calculation (Reservoir Sampling)
Due to the massive dataset containing nearly 17.5 million sequences, calculating full biochemical properties is extremely time-consuming. We employ random reservoir sampling (N≈100,000) to calculate representative biochemical characteristics.

In [ ]:
faa_files = glob.glob(os.path.join(PRED_DIR, "*_enzymes.faa"))
SAMPLE_RATE = 100000 / 17500000.0

sampled_seqs = []
lengths = []

def clean_seq(seq):
    return ''.join([aa for aa in seq if aa in "ACDEFGHIKLMNPQRSTVWY"])

for faa in tqdm(faa_files, desc="Sampling FASTA"):
    with open(faa, 'r') as f:
        seq_chunks = []
        for line in f:
            if line.startswith(">"):
                if seq_chunks:
                    seq = "".join(seq_chunks)
                    lengths.append(len(seq))
                    if random.random() < SAMPLE_RATE:
                        sampled_seqs.append(seq)
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        if seq_chunks:
            seq = "".join(seq_chunks)
            lengths.append(len(seq))
            if random.random() < SAMPLE_RATE:
                sampled_seqs.append(seq)

print(f"Full length extraction complete, total {len(lengths):,} sequences.")
print(f"Successfully sampled {len(sampled_seqs):,} sequences for biochemical calculation.")

pis = []
mws = []

for seq in tqdm(sampled_seqs, desc="Computing ProtParam"):
    clean_s = clean_seq(seq)
    if not clean_s:
        continue
    pa = ProteinAnalysis(clean_s)
    try:
        pis.append(pa.isoelectric_point())
        mws.append(pa.molecular_weight() / 1000.0) # kDa
    except Exception:
        pass

## 5. Visualization of Sequence Length, Molecular Weight, and Isoelectric Point

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.despine()

# 1. Length Distribution
lengths_arr = np.array(lengths)
p99_len = np.percentile(lengths_arr, 99)
sns.histplot(lengths_arr[lengths_arr <= p99_len], bins=80, kde=True, color="#00A087", ax=axes[0, 0])
axes[0, 0].axvline(np.median(lengths_arr), color='red', linestyle='dashed', label=f'Median: {np.median(lengths_arr):.0f}')
axes[0, 0].set_title('Enzyme Sequence Length Distribution')
axes[0, 0].set_xlabel('Length (AA)')
axes[0, 0].legend()

# 2. Prediction Probability
sns.histplot(all_probabilities, bins=50, kde=True, color="#3C5488", ax=axes[0, 1])
axes[0, 1].axvline(np.median(all_probabilities), color='red', linestyle='dashed', label=f'Median: {np.median(all_probabilities):.3f}')
axes[0, 1].set_title('Prediction Probability Distribution (Sampled)')
axes[0, 1].set_xlabel('XGBoost Probability')
axes[0, 1].legend()

# 3. Isoelectric Point (pI)
sns.histplot(pis, bins=50, kde=True, color="#E64B35", ax=axes[1, 0])
axes[1, 0].set_title(f'Isoelectric Point (pI) Distribution')
axes[1, 0].set_xlabel('pI')

# 4. Molecular Weight
p99_mw = np.percentile(mws, 99)
filtered_mws = [m for m in mws if m <= p99_mw]
sns.histplot(filtered_mws, bins=50, kde=True, color="#8491B4", ax=axes[1, 1])
axes[1, 1].set_title('Molecular Weight Distribution')
axes[1, 1].set_xlabel('Molecular Weight (kDa)')

plt.tight_layout()
plt.show()